# 📊 Notebook 1 — Ingesta de Datos Reales con yfinance
**Sistema Ernesto Investing AI — Backend con IA**

**Curso:** iDeSo · UNMSM · FISI  
**Docente:** Prof. E. D. Cancho-Rodríguez, MBA (GWU)  
**Objetivo:** Descargar datos OHLCV reales de 5 empresas mineras con operaciones en Perú,
calcular indicadores técnicos y exportar un JSON compatible con el frontend
`ernesto_investing_dashboard_mercado.html`.

---
### 🏢 Tickers del estudio:
| Ticker | Empresa | Bolsa |
|--------|---------|-------|
| FSM | Fortuna Silver Mines Inc. | NYSE |
| VOLCABC1.LM | Volcan Compañía Minera S.A.A. | BVL |
| ABX.TO | Barrick Gold Corporation | TSX |
| BVN | Compañía de Minas Buenaventura S.A.A. | NYSE |
| BHP | BHP Group Limited | NYSE |

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — INSTALACIÓN DE LIBRERÍAS                            ║
# ║  Se instalan las dependencias necesarias para el notebook.      ║
# ╚══════════════════════════════════════════════════════════════════╝

!pip install yfinance ta --quiet
print("✅ Librerías instaladas correctamente.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — IMPORTACIONES                                        ║
# ║  Se importan todas las librerías que se usarán en el notebook.  ║
# ╚══════════════════════════════════════════════════════════════════╝

import yfinance as yf          # Descarga de datos financieros de Yahoo Finance
import pandas as pd             # Manipulación de datos en DataFrames
import numpy as np              # Operaciones matemáticas y vectoriales
import json                     # Exportación de datos a formato JSON
import warnings                 # Control de advertencias
from datetime import datetime, timedelta  # Manejo de fechas

# Importaciones de la librería 'ta' para indicadores técnicos
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

# Suprimimos advertencias de deprecación para mayor limpieza en la salida
warnings.filterwarnings('ignore')

print("✅ Importaciones completadas correctamente.")
print(f"   yfinance: {yf.__version__}")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — CONFIGURACIÓN DE TICKERS Y PERÍODO TEMPORAL          ║
# ║  Se definen los 5 tickers del estudio y el rango de fechas.     ║
# ╚══════════════════════════════════════════════════════════════════╝

# --- Tickers de empresas mineras con operaciones en Perú ---
TICKERS = [
    'FSM',          # Fortuna Silver Mines — NYSE
    'VOLCABC1.LM',  # Volcan Compañía Minera — BVL
    'ABX.TO',       # Barrick Gold Corporation — TSX
    'BVN',          # Compañía de Minas Buenaventura — NYSE
    'BHP'           # BHP Group Limited — NYSE
]

# Metadatos de cada empresa (para el frontend)
METADATA_EMPRESAS = {
    'FSM':         {'nombre': 'Fortuna Silver Mines Inc.',             'corto': 'Fortuna Silver', 'bolsa': 'NYSE'},
    'VOLCABC1.LM': {'nombre': 'Volcan Compañía Minera S.A.A.',        'corto': 'Volcan',         'bolsa': 'BVL'},
    'ABX.TO':      {'nombre': 'Barrick Gold Corporation',              'corto': 'Barrick Gold',   'bolsa': 'TSX'},
    'BVN':         {'nombre': 'Compañía de Minas Buenaventura S.A.A.','corto': 'Buenaventura',   'bolsa': 'NYSE'},
    'BHP':         {'nombre': 'BHP Group Limited',                     'corto': 'BHP',            'bolsa': 'NYSE'}
}

# --- Período de descarga: últimos 2 años ---
FECHA_FIN   = datetime.today()
FECHA_INICIO = FECHA_FIN - timedelta(days=730)  # 2 años = 730 días

START_DATE = FECHA_INICIO.strftime('%Y-%m-%d')
END_DATE   = FECHA_FIN.strftime('%Y-%m-%d')

print(f"📅 Período de descarga: {START_DATE}  →  {END_DATE}")
print(f"📌 Tickers configurados: {TICKERS}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 4 — DESCARGA DE DATOS OHLCV CON MANEJO DE ERRORES        ║
# ║  Se descarga Open, High, Low, Close, Volume para cada ticker.   ║
# ╚══════════════════════════════════════════════════════════════════╝

def descargar_datos_ticker(ticker: str, start: str, end: str) -> pd.DataFrame | None:
    """
    Descarga datos OHLCV de Yahoo Finance para un ticker dado.

    Parámetros:
        ticker (str): Símbolo bursátil del activo.
        start  (str): Fecha de inicio en formato 'YYYY-MM-DD'.
        end    (str): Fecha de fin en formato 'YYYY-MM-DD'.

    Retorna:
        pd.DataFrame con columnas OHLCV, o None si la descarga falla.
    """
    try:
        print(f"  ⬇️  Descargando {ticker}...", end=' ')
        df = yf.download(
            ticker,
            start=start,
            end=end,
            auto_adjust=True,   # Ajusta precios por splits y dividendos
            progress=False
        )

        # Verificar que se obtuvieron datos
        if df.empty:
            print(f"⚠️  Sin datos disponibles para {ticker}.")
            return None

        # Aplanar columnas MultiIndex que genera yfinance (si aplica)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Renombrar columnas al estándar interno del proyecto
        df = df.rename(columns={
            'Open':   'open',
            'High':   'high',
            'Low':    'low',
            'Close':  'close',
            'Volume': 'volume'
        })

        # Eliminar filas con valores nulos en precios esenciales
        df = df.dropna(subset=['open', 'high', 'low', 'close'])

        # Resetear índice para que la fecha quede como columna
        df = df.reset_index()
        df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

        print(f"✅ {len(df)} registros descargados.")
        return df[['Date', 'open', 'high', 'low', 'close', 'volume']]

    except Exception as e:
        print(f"❌ Error al descargar {ticker}: {e}")
        return None


# --- Ejecutar descarga para todos los tickers ---
print("🚀 Iniciando descarga de datos OHLCV desde Yahoo Finance...\n")
datos_crudos = {}   # Diccionario {ticker: DataFrame}

for ticker in TICKERS:
    df = descargar_datos_ticker(ticker, START_DATE, END_DATE)
    if df is not None:
        datos_crudos[ticker] = df

print(f"\n📦 Tickers descargados exitosamente: {list(datos_crudos.keys())}")
print(f"⚠️  Tickers sin datos: {[t for t in TICKERS if t not in datos_crudos]}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — CÁLCULO DE INDICADORES TÉCNICOS                      ║
# ║  SMA-20, SMA-50, EMA-12, EMA-26, RSI-14, MACD, Bollinger Bands ║
# ╚══════════════════════════════════════════════════════════════════╝

def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula todos los indicadores técnicos requeridos por el frontend.

    Indicadores calculados:
        - SMA-20  : Media Móvil Simple de 20 períodos
        - SMA-50  : Media Móvil Simple de 50 períodos
        - EMA-12  : Media Móvil Exponencial de 12 períodos
        - EMA-26  : Media Móvil Exponencial de 26 períodos
        - RSI-14  : Índice de Fuerza Relativa de 14 períodos
        - MACD    : Línea MACD (EMA12 - EMA26)
        - MACD Signal: Media EMA-9 sobre la línea MACD
        - MACD Hist: Histograma MACD
        - BB Upper: Banda Superior de Bollinger (SMA20 + 2σ)
        - BB Lower: Banda Inferior de Bollinger (SMA20 - 2σ)

    Parámetro:
        df (pd.DataFrame): DataFrame con columna 'close'.

    Retorna:
        pd.DataFrame con todas las columnas de indicadores añadidas.
    """
    close_series = df['close']

    # --- Medias Móviles Simples (SMA) ---
    df['sma_20'] = SMAIndicator(close=close_series, window=20).sma_indicator().round(4)
    df['sma_50'] = SMAIndicator(close=close_series, window=50).sma_indicator().round(4)

    # --- Medias Móviles Exponenciales (EMA) ---
    df['ema_12'] = EMAIndicator(close=close_series, window=12).ema_indicator().round(4)
    df['ema_26'] = EMAIndicator(close=close_series, window=26).ema_indicator().round(4)

    # --- RSI (Índice de Fuerza Relativa) ---
    df['rsi_14'] = RSIIndicator(close=close_series, window=14).rsi().round(4)

    # --- MACD (Moving Average Convergence Divergence) ---
    macd_obj = MACD(
        close=close_series,
        window_slow=26,
        window_fast=12,
        window_sign=9
    )
    df['macd']        = macd_obj.macd().round(4)          # Línea MACD
    df['macd_signal'] = macd_obj.macd_signal().round(4)   # Línea de señal
    df['macd_hist']   = macd_obj.macd_diff().round(4)     # Histograma

    # --- Bollinger Bands (BB) ---
    bb_obj = BollingerBands(close=close_series, window=20, window_dev=2)
    df['bb_upper'] = bb_obj.bollinger_hband().round(4)    # Banda superior
    df['bb_middle']= bb_obj.bollinger_mavg().round(4)     # Media (= SMA-20)
    df['bb_lower'] = bb_obj.bollinger_lband().round(4)    # Banda inferior

    return df


def generar_senales_cruce_sma(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera señales BUY/SELL basadas en el cruce de SMA-20 y SMA-50.

    Lógica:
        - BUY  : SMA-20 cruza hacia arriba a SMA-50 (cruce dorado)
        - SELL : SMA-20 cruza hacia abajo a SMA-50 (cruce de la muerte)

    Parámetro:
        df (pd.DataFrame): DataFrame con columnas 'sma_20' y 'sma_50'.

    Retorna:
        pd.DataFrame con columna 'signal' añadida ('BUY', 'SELL' o None).
    """
    signals = [None]  # El primer día no tiene señal (sin período anterior)

    for i in range(1, len(df)):
        sma20_hoy   = df['sma_20'].iloc[i]
        sma50_hoy   = df['sma_50'].iloc[i]
        sma20_ayer  = df['sma_20'].iloc[i - 1]
        sma50_ayer  = df['sma_50'].iloc[i - 1]

        # Verificar que todos los valores sean válidos (no NaN)
        if any(pd.isna(v) for v in [sma20_hoy, sma50_hoy, sma20_ayer, sma50_ayer]):
            signals.append(None)
        elif sma20_hoy > sma50_hoy and sma20_ayer <= sma50_ayer:
            signals.append('BUY')   # Cruce dorado: señal de compra
        elif sma20_hoy < sma50_hoy and sma20_ayer >= sma50_ayer:
            signals.append('SELL')  # Cruce de la muerte: señal de venta
        else:
            signals.append(None)

    df['signal'] = signals
    return df


# --- Aplicar indicadores a todos los tickers descargados ---
print("⚙️  Calculando indicadores técnicos...\n")
datos_con_indicadores = {}

for ticker, df in datos_crudos.items():
    print(f"  📈 Procesando {ticker}...", end=' ')
    df_ind = calcular_indicadores(df.copy())
    df_ind = generar_senales_cruce_sma(df_ind)
    datos_con_indicadores[ticker] = df_ind
    n_buy  = (df_ind['signal'] == 'BUY').sum()
    n_sell = (df_ind['signal'] == 'SELL').sum()
    print(f"✅  ({n_buy} BUY, {n_sell} SELL)")

print("\n✅ Indicadores calculados para todos los tickers.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 6 — TRANSFORMACIÓN A ESTRUCTURA JSON DEL FRONTEND         ║
# ║  Se construye el JSON exacto que consume el dashboard HTML.     ║
# ╚══════════════════════════════════════════════════════════════════╝

def safe_float(value) -> float | None:
    """
    Convierte un valor a float seguro para JSON.
    Retorna None si el valor es NaN o infinito.
    """
    try:
        f = float(value)
        return None if (pd.isna(f) or np.isinf(f)) else round(f, 4)
    except (TypeError, ValueError):
        return None


def construir_json_ticker(ticker: str, df: pd.DataFrame) -> dict:
    """
    Transforma un DataFrame con indicadores en el diccionario JSON
    compatible con el frontend ernesto_investing_dashboard_mercado.html.

    Estructura del JSON por ticker:
    {
        "ticker": "BVN",
        "nombre": "Compañía de Minas Buenaventura S.A.A.",
        "bolsa": "NYSE",
        "fechas": [...],
        "opens": [...], "highs": [...], "lows": [...], "closes": [...],
        "volumes": [...],
        "indicadores": {
            "sma_20": [...], "sma_50": [...],
            "ema_12": [...], "ema_26": [...],
            "rsi_14": [...],
            "macd": [...], "macd_signal": [...], "macd_hist": [...],
            "bb_upper": [...], "bb_middle": [...], "bb_lower": [...]
        },
        "senales": [...],
        "ultima_senal": "BUY" | "SELL",
        "precio_ultimo": 16.43,
        "cambio_dia": 0.27,
        "cambio_pct": 1.67,
        "precio_max": 18.50,
        "precio_min": 13.20,
        "volumen_promedio": 1234567
    }
    """
    meta = METADATA_EMPRESAS.get(ticker, {'nombre': ticker, 'corto': ticker, 'bolsa': 'N/A'})

    # Extraer arrays de precios y volumen
    fechas  = df['Date'].tolist()
    opens   = [safe_float(v) for v in df['open'].tolist()]
    highs   = [safe_float(v) for v in df['high'].tolist()]
    lows    = [safe_float(v) for v in df['low'].tolist()]
    closes  = [safe_float(v) for v in df['close'].tolist()]
    volumes = [int(v) if not pd.isna(v) else 0 for v in df['volume'].tolist()]

    # Extraer indicadores
    indicadores = {
        'sma_20':      [safe_float(v) for v in df['sma_20'].tolist()],
        'sma_50':      [safe_float(v) for v in df['sma_50'].tolist()],
        'ema_12':      [safe_float(v) for v in df['ema_12'].tolist()],
        'ema_26':      [safe_float(v) for v in df['ema_26'].tolist()],
        'rsi_14':      [safe_float(v) for v in df['rsi_14'].tolist()],
        'macd':        [safe_float(v) for v in df['macd'].tolist()],
        'macd_signal': [safe_float(v) for v in df['macd_signal'].tolist()],
        'macd_hist':   [safe_float(v) for v in df['macd_hist'].tolist()],
        'bb_upper':    [safe_float(v) for v in df['bb_upper'].tolist()],
        'bb_middle':   [safe_float(v) for v in df['bb_middle'].tolist()],
        'bb_lower':    [safe_float(v) for v in df['bb_lower'].tolist()]
    }

    # Señales de cruce
    senales = df['signal'].tolist()  # Lista con 'BUY', 'SELL' o None

    # Calcular última señal no nula
    ultima_senal = 'BUY'  # Valor por defecto
    for s in reversed(senales):
        if s is not None:
            ultima_senal = s
            break

    # Métricas de resumen del activo
    closes_validos = [c for c in closes if c is not None]
    precio_ultimo  = closes_validos[-1] if closes_validos else None
    precio_ant     = closes_validos[-2] if len(closes_validos) >= 2 else precio_ultimo

    cambio_dia  = round(precio_ultimo - precio_ant, 4) if (precio_ultimo and precio_ant) else 0
    cambio_pct  = round((cambio_dia / precio_ant) * 100, 2) if precio_ant else 0
    precio_max  = round(max(closes_validos), 4) if closes_validos else None
    precio_min  = round(min(closes_validos), 4) if closes_validos else None
    vol_promedio = int(np.mean([v for v in volumes if v > 0])) if volumes else 0

    return {
        'ticker':           ticker,
        'nombre':           meta['nombre'],
        'nombre_corto':     meta['corto'],
        'bolsa':            meta['bolsa'],
        'periodo_inicio':   fechas[0] if fechas else None,
        'periodo_fin':      fechas[-1] if fechas else None,
        'total_registros':  len(fechas),
        'fechas':           fechas,
        'opens':            opens,
        'highs':            highs,
        'lows':             lows,
        'closes':           closes,
        'volumes':          volumes,
        'indicadores':      indicadores,
        'senales':          senales,
        'ultima_senal':     ultima_senal,
        'precio_ultimo':    precio_ultimo,
        'cambio_dia':       cambio_dia,
        'cambio_pct':       cambio_pct,
        'precio_max':       precio_max,
        'precio_min':       precio_min,
        'volumen_promedio': vol_promedio
    }


# --- Construir el JSON completo con todos los tickers ---
print("🔧 Construyendo estructura JSON para el frontend...\n")
json_output = {
    'metadata': {
        'fuente':            'Yahoo Finance (yfinance)',
        'generado_en':       datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'periodo_inicio':    START_DATE,
        'periodo_fin':       END_DATE,
        'tickers_solicitados': TICKERS,
        'tickers_descargados': list(datos_con_indicadores.keys()),
        'notebook':          'Notebook 1 — Ingesta de Datos con yfinance',
        'proyecto':          'Ernesto Investing AI — iDeSo UNMSM FISI',
        'frontend_destino':  'ernesto_investing_dashboard_mercado.html'
    },
    'datos': {}
}

for ticker, df in datos_con_indicadores.items():
    print(f"  🔁 Serializando {ticker}...", end=' ')
    json_output['datos'][ticker] = construir_json_ticker(ticker, df)
    print(f"✅  ({json_output['datos'][ticker]['total_registros']} registros)")

print("\n✅ Estructura JSON construida correctamente.")

# --- Vista previa del JSON: resumen de precios ---
print("\n📊 Resumen de precios actuales:")
print(f"{'Ticker':<15} {'Precio':>8} {'Cambio%':>9} {'Señal':>6} {'Registros':>10}")
print("-" * 55)
for ticker, data in json_output['datos'].items():
    precio  = f"${data['precio_ultimo']:.2f}" if data['precio_ultimo'] else 'N/A'
    cambio  = f"{data['cambio_pct']:+.2f}%" if data['cambio_pct'] is not None else 'N/A'
    senal   = data['ultima_senal']
    regs    = data['total_registros']
    print(f"{ticker:<15} {precio:>8} {cambio:>9} {senal:>6} {regs:>10}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 7 — EXPORTACIÓN A ARCHIVO datos_mercado.json             ║
# ║  Se guarda el JSON final que consumirá el frontend HTML.        ║
# ╚══════════════════════════════════════════════════════════════════╝

# Nombre del archivo de salida (contrato de datos con el frontend)
OUTPUT_FILENAME = 'datos_mercado.json'

# Serializar y guardar el archivo JSON con indentación legible
with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as f:
    json.dump(json_output, f, ensure_ascii=False, indent=2, default=str)

# --- Verificación del archivo exportado ---
import os
file_size_kb = os.path.getsize(OUTPUT_FILENAME) / 1024

print("═" * 60)
print("  ✅  EXPORTACIÓN COMPLETADA EXITOSAMENTE")
print("═" * 60)
print(f"  📄 Archivo generado : {OUTPUT_FILENAME}")
print(f"  📦 Tamaño           : {file_size_kb:.1f} KB")
print(f"  🏢 Tickers incluidos: {list(json_output['datos'].keys())}")
print(f"  📅 Generado el      : {json_output['metadata']['generado_en']}")
print(f"  🎯 Frontend destino : {json_output['metadata']['frontend_destino']}")
print("═" * 60)

# --- Descarga automática en Google Colab ---
try:
    from google.colab import files
    files.download(OUTPUT_FILENAME)
    print("\n⬇️  El archivo se está descargando en tu equipo...")
except ImportError:
    # Si no estamos en Colab, indicar la ruta del archivo
    print(f"\n📁 Archivo guardado en: {os.path.abspath(OUTPUT_FILENAME)}")

# --- Vista previa de las primeras claves del JSON ---
print("\n🔍 Vista previa del JSON (estructura de metadatos):")
preview = {
    'metadata': json_output['metadata'],
    'tickers_en_datos': list(json_output['datos'].keys()),
    'campos_por_ticker': list(json_output['datos'][list(json_output['datos'].keys())[0]].keys())
                          if json_output['datos'] else []
}
print(json.dumps(preview, indent=2, ensure_ascii=False))